## Catalogs , Schemas  ,Tables

In [0]:
-- to see catalogs
SHOW CATALOGS;
SHOW CATALOGS like 'dev*';
SHOW CATALOGS like '*m'


catalog
deltalake1
dlt
ext_catalog
man_catalog
muaazdatabricks_7405611966038211
samples
system
test1


In [0]:
-- Info about catalog  
DESCRIBE CATALOG EXTENDED democatalog

info_name,info_value
Catalog Name,democatalog
Comment,
Owner,muaazmuzammil69@gmail.com
Catalog Type,Regular
Created By,muaazmuzammil69@gmail.com
Created At,2026-05-12 AD at 14:32:54 UTC
Updated By,muaazmuzammil69@gmail.com
Updated At,2026-05-17 AD at 14:04:29 UTC
Storage Root,
Storage Location,


In [0]:
-- to drop a catalog
DROP CATALOG deltalake1 CASCADE

In [0]:
-- to see schemas
SHOW SCHEMAS IN catalog1;
SHOW SCHEMAS IN catalog1 like 'raw*';
SHOW SCHEMAS IN catalog1 like '*m'


In [0]:
-- info about schema
DESCRIBE SCHEMA EXTENDED democatalog.schema1

database_description_item,database_description_value
Catalog Name,democatalog
Namespace Name,schema1
Comment,
Location,
Owner,muaazmuzammil69@gmail.com
Properties,
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


In [0]:
-- to see tables 
SHOW TABLES IN catalog1.schema1;
SHOW TABLES IN catalog1.schema1 like 'sale*';
SHOW TABLES IN catalog1.schema1 like '*m'

-- for python
-- spark.catalog.tbaleExits("catalog1.schema1.table1")

## Managed Catalog Managed schema Managed table

In [0]:
-- If we do not provide a catalog location while creating it, it defaults to the metastore assigned location.
-- Similarly, if we do not provide a schema location under a catalog, it will default to the catalog’s storage location.
-- Similarly, if we create a table and do not assign a location, it will use the schema location. 
-- In short, if we do not specify a location at the catalog level, it will default to the metastore storage location. Likewise, at lower levels of the hierarchy, if no location is specified, the object will be created in the parent’s storage location.

In [0]:
-- Metastore level storage location.
-- The metastore is associated with the underlying data lake storage.

-- This catalog will be created in the metastore's managed storage location for managed objects.
CREATE CATALOG man_catalog

In [0]:
CREATE schema man_catalog.man_schema

In [0]:
-- when we create tables in unity catalog we use 3 level namespace catalog.schema.table

CREATE TABLE  man_catalog.man_schema.man_table 
(
  id INT, 
  name STRING
) 
USING  DELTA;

In [0]:
-- to see info of tables
DESCRIBE EXTENDED man_catalog.man_schema.man_table

In [0]:
INSERT INTO man_catalog.man_schema.man_table VALUES (1, 'John'), (2, 'Jane')


num_affected_rows,num_inserted_rows
2,2


## External Location Catalog (Overrides Metastore Default Location, Metastore Managed) Managed schema Managed table

In [0]:
-- In Unity Catalog, you cannot create an "external catalog" in the sense of bypassing Unity Catalog.
-- All catalogs are managed by Unity Catalog; there is no concept of an unmanaged or external catalog.

-- Catalogs and schemas are always metadata managed by Unity Catalog.
-- Tables can be either managed or external depending on whether a custom storage location is specified.

-- The metastore is associated with the underlying data lake storage.
-- A catalog is configured with a storage location within the data lake for managed objects.

In [0]:
-- This creates a catalog with a managed storage location different from the metastore default location.
-- It is still a Unity Catalog managed catalog; only the storage location is customized.
-- The storage hierarchy for managed objects will be based on this specified location.

-- "Managed" is used because tables created under this catalog are still Unity Catalog managed objects.
CREATE CATALOG ext_catalog
MANAGED LOCATION 'abfss://unitycatalog@muaazexternalstorage.dfs.core.windows.net/Catalogs'

In [0]:
CREATE schema ext_catalog.man_schema


In [0]:
CREATE TABLE  ext_catalog.man_schema.man_table 
(
  id INT, 
  name STRING
) 
USING  DELTA;

## External Catalog (Overrides Metastore Default Location, Metastore Managed) Eternal schema(Overrides Metastore Default Location, Metastore Managed)  Managed table 

In [0]:
-- The metastore is associated with the underlying data lake storage system.
-- A catalog is configured with a storage location within the data lake (for managed objects).
-- A schema can also have an optional storage location within the data lake.

In [0]:
-- This will create an external schema under the specified external catalog.

CREATE schema ext_catalog.ext_schema
MANAGED Location  'abfss://unitycatalog@muaazexternalstorage.dfs.core.windows.net/ext_schema'

In [0]:
CREATE TABLE  ext_catalog.ext_schema.man_table 
(
  id INT, 
  name STRING
) 
USING  DELTA;

## External Catalog (Overrides Metastore Default Location, Metastore Managed) Eternal schema(Overrides Metastore Default Location, Metastore Managed) Eternal table

In [0]:
-- Object-level storage (external tables)

-- The metastore is associated with the underlying data lake storage system.
-- A catalog is configured with a storage location in the data lake.
-- A schema may optionally define its own storage location in the data lake.
-- A table stores data in the data lake; it would be external because location is specified.

In [0]:
-- will create external table with managed catalog and managed schema
CREATE TABLE  man_catalog.man_schema.ext_table 
(
  id INT, 
  name STRING
) 
USING  DELTA
LOCATION  'abfss://unitycatalog@muaazexternalstorage.dfs.core.windows.net/external_table'

## Table Drop

In [0]:
-- DROP managed table: it does not immediately remove data from storage.
-- Data is retained for a limited retention period (e.g., 7 days), during which the table can be recovered using UNDROP.

DROP TABLE man_catalog.man_schema.man_table

In [0]:
-- to see tables which has been dropped and in retention period
Show tables dropped IN man_catalog.man_schema

In [0]:
-- UNDROP managed table
-- When a managed table is dropped, the metastore does not immediately remove the data.
-- The data is retained for a limited period (e.g., up to 7 days), allowing recovery using the UNDROP command.
-- UNDROP can be used to restore both managed and (in supported cases) external tables depending on the platform.


UNDROP TABLE man_catalog.man_schema.man_table

## VIEWs

In [0]:
-- Permanent View
-- A view does not physically store data; it is a logical abstraction over a query.
-- Data is always read from the underlying base tables when the view is queried.

CREATE VIEW man_catalog.man_schema.man_view AS SELECT * FROM man_catalog.man_schema.man_table

In [0]:
-- Temporary View
-- In this case, if the notebook/session is detached or restarted, the view will be lost.
-- A temporary view exists only for the duration of the session.

CREATE OR REPLACE TEMP VIEW temp_view AS SELECT * FROM man_catalog.man_schema.man_table

## VOLUMEs

In [0]:
%python
# Creating directory for volume
dbutils.fs.mkdirs("abfss://unitycatalog@muaazexternalstorage.dfs.core.windows.net/volumes")

True

In [0]:
-- will create external volume on that location
CREATE EXTERNAL VOLUME man_catalog.man_schema.my_volume
LOCATION 'abfss://unitycatalog@muaazexternalstorage.dfs.core.windows.net/volumes'

In [0]:
DESCRIBE VOLUME man_catalog.man_schema.my_volume

In [0]:
%python
# -- copy file from source to vlume

dbutils.fs.cp('abfss://bronze@muaazexternalstorage.dfs.core.windows.net/source/catalog_report','abfss://unitycatalog@muaazexternalstorage.dfs.core.windows.net/volumes/Sales')

In [0]:
SELECT * FROM csv.`/Volumes/man_catalog/man_schema/my_volume/catalog_report.csv`

_c0,_c1,_c2,_c3,_c4,_c5,_c6
catalog,schema,table,row_count,path,uuid,schema_id
migration,working,table2,3,abfss://migrationcatalog@muaazexternalstorage.dfs.core.windows.net/working/__unitystorage/schemas/0f4d3cdc-4001-4c29-8062-b43d0e244d03/tables/721e2713-5af4-4332-b39a-c3673351e964,721e2713-5af4-4332-b39a-c3673351e964,0f4d3cdc-4001-4c29-8062-b43d0e244d03
migration,working,table1,2,abfss://migrationcatalog@muaazexternalstorage.dfs.core.windows.net/working/__unitystorage/schemas/0f4d3cdc-4001-4c29-8062-b43d0e244d03/tables/92987ac7-7d76-4581-94a2-9be6f53bf18d,92987ac7-7d76-4581-94a2-9be6f53bf18d,0f4d3cdc-4001-4c29-8062-b43d0e244d03
migration,working,table3,4,abfss://migrationcatalog@muaazexternalstorage.dfs.core.windows.net/working/__unitystorage/schemas/0f4d3cdc-4001-4c29-8062-b43d0e244d03/tables/d2f19bfb-c596-4256-b33a-c11a92305f64,d2f19bfb-c596-4256-b33a-c11a92305f64,0f4d3cdc-4001-4c29-8062-b43d0e244d03
migration,external,table1,2,abfss://migrationcatalog@muaazexternalstorage.dfs.core.windows.net/external/__unitystorage/schemas/4413ab73-1ac1-456d-9438-0e79149641b8/tables/0051ea3d-3d1c-4d1a-9146-b16fbe9a8f8a,0051ea3d-3d1c-4d1a-9146-b16fbe9a8f8a,4413ab73-1ac1-456d-9438-0e79149641b8
migration,external,table2,3,abfss://migrationcatalog@muaazexternalstorage.dfs.core.windows.net/external/__unitystorage/schemas/4413ab73-1ac1-456d-9438-0e79149641b8/tables/a166bee3-b26b-408f-9154-0877e8df8fea,a166bee3-b26b-408f-9154-0877e8df8fea,4413ab73-1ac1-456d-9438-0e79149641b8
migration,external,table3,4,abfss://migrationcatalog@muaazexternalstorage.dfs.core.windows.net/external/__unitystorage/schemas/4413ab73-1ac1-456d-9438-0e79149641b8/tables/d72c7ee4-6022-4856-80e0-53ea40348581,d72c7ee4-6022-4856-80e0-53ea40348581,4413ab73-1ac1-456d-9438-0e79149641b8


In [0]:
DROP VOLUME man_catalog.man_schema.my_volume

## Functions

In [0]:
-- Unity catalog functions
-- Will create external table with managed catalog and managed schema
CREATE TABLE  ext_catalog.ext_schema.exployees 
(
  id INT, 
  name STRING
) 
USING  DELTA;

INSERT INTO ext_catalog.ext_schema.exployees VALUES (1,'John'),(2,'Jane'),(3,'Bob'),(4,'Alice'),(5,'Charlie'),(6,'Dave'),(7,'Eve'),(8,'Frank'),(9,'Grace'),(10,'Hannah')

num_affected_rows,num_inserted_rows
10,10


In [0]:
-- Create a masking function using group based access control
-- This function returns the real value for users in a specific group,
-- otherwise it masks the value.

CREATE FUNCTION ext_catalog.ext_schema.masking( id INT)
RETURN CASE WHEN is_account_group_member("mmuaaaz") THEN id ELSE '**********' END

In [0]:
-- apply function to table

Alter table ext_catalog.ext_schema.exployees alter column id set MASK ext_catalog.ext_schema.masking
-- apply function to view


In [0]:
-- If the user is not a member of the specified group in the masking function,
-- they will not see the actual value and instead will see '**********'.

SELECT * FROM ext_catalog.ext_schema.exployees 

## External Credential Creation

In [0]:
CREATE EXTERNAL CREDENTIAL 'name'
WITH AZURE_MANAGED_IDENTITY (
    ACCESS_CONNECTOR_ID = '/subscriptions/04aa59aa-9e01-4d29-9b54-64a7e180ed83/resourceGroups/databricks-practice/providers/Microsoft.Databricks/accessConnectors/muaazconnector'
);

SHOW EXTERNAL CREDENTIALS;

## External Location Creation

In [0]:
CREATE EXTERNAL LOCATION 'name'
URL 'abfss://<container>@<storage-account>.dfs.core.windows.net'
WITH (STORAGE CREDENTIAL 'credential_name');

SHOW EXTERNAL LOCATIONS;

## Object Level Permissions

In [0]:
-- to check permission on an object , this will show all permissions to every user on metastore
SHOW GRANT ON METASTORE

Principal,ActionType,ObjectType,ObjectKey
_workspace_admins_workspace_7474651555292993,CREATE CATALOG,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE PROVIDER,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE SHARE,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE EXTERNAL LOCATION,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE CONNECTION,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE SERVICE CREDENTIAL,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE RECIPIENT,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE CLEAN ROOM,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
_workspace_admins_workspace_7474651555292993,CREATE STORAGE CREDENTIAL,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde
account users,USE MARKETPLACE ASSETS,METASTORE,10531913-8f10-4baf-849b-f92de7e45dde


In [0]:
-- This operation is typically performed by a metastore admin.
-- It grants the user permission to create catalogs within the metastore.

GRANT CREATE CATALOG ON METASTORE TO `muaaz123@gmail.com`;